# Module 6.2: Factor-Augmented Vector Autoregression (FAVAR)

## Learning Objectives

By completing this notebook, you will:

1. **Understand** the FAVAR framework and its advantages over standard VAR
2. **Estimate** FAVAR models using two-step and joint approaches
3. **Identify** structural shocks using Cholesky and sign restrictions
4. **Compute** impulse response functions (IRFs) for factors and observables
5. **Analyze** monetary policy transmission through factors
6. **Visualize** the impact of structural shocks across many variables

## Prerequisites

- VAR models and impulse response functions
- Factor models and PCA (Modules 1-3)
- Diffusion index forecasting (Module 6.1)
- Structural identification (Cholesky decomposition)

## Estimated Time: 120-150 minutes

---

## Setup and Imports

We'll use tools for VAR estimation, factor extraction, and IRF computation.

In [ ]:
# Purpose: Import libraries for FAVAR estimation and analysis
# Key Concept: FAVAR combines factor models with structural VAR

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from scipy import linalg
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(2024)

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11

print("Libraries imported successfully!")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

---

# Part 1: FAVAR Framework

## 1.1 Motivation: Standard VAR Limitations

**Problem with standard VAR:**
- Curse of dimensionality: $K$ variables, $p$ lags → $K^2 p$ parameters
- Example: 10 variables, 4 lags → 400 parameters!
- Limited information: Can only include ~5-10 variables
- Variable selection: Which indicators to include?

**FAVAR solution (Bernanke, Boivin & Eliasz, 2005):**
- Extract **factors** from large panel (100+ variables)
- Estimate **VAR** with factors + key observed variables
- Trace impact through factors to **all** original variables

## FAVAR Specification

**Measurement equation:**
$$X_t = \Lambda^F F_t + \Lambda^Y Y_t + e_t$$

where:
- $X_t$ (N×1): Large panel of informational variables
- $F_t$ (r×1): Unobserved factors
- $Y_t$ (m×1): Observed policy/structural variables (e.g., Fed Funds rate)
- $\Lambda^F$ (N×r), $\Lambda^Y$ (N×m): Loadings
- $e_t$: Idiosyncratic errors

**Transition equation (VAR):**
$$\begin{bmatrix} F_t \\ Y_t \end{bmatrix} = \Phi(L) \begin{bmatrix} F_{t-1} \\ Y_{t-1} \end{bmatrix} + u_t$$

**Key features:**
1. Factors $F_t$ and observables $Y_t$ jointly evolve as VAR
2. Factors summarize information from 100+ variables
3. Can identify structural shocks and trace to all observables

## 1.2 Generate FAVAR Data

We'll simulate a monetary policy example:
- **Large panel** (50 macro variables): IP, employment, prices, etc.
- **Observed policy variable:** Federal Funds rate
- **Structural shock:** Monetary policy surprise

In [ ]:
# Purpose: Generate FAVAR data with monetary policy structure
# Key Concept: Policy rate affects economy through common factors

def generate_favar_data(T=300, N=50, r=3, seed=42):
    """
    Generate synthetic FAVAR data for monetary policy analysis.
    
    Parameters
    ----------
    T : int
        Number of time periods
    N : int
        Number of informational variables
    r : int
        Number of factors
    seed : int
        Random seed
    
    Returns
    -------
    X : ndarray, shape (T, N)
        Panel of macro variables
    Y_policy : ndarray, shape (T,)
        Policy rate (Fed Funds)
    F_true : ndarray, shape (T, r)
        True factors
    var_names : list
        Variable names
    dates : DatetimeIndex
        Time index
    """
    np.random.seed(seed)
    
    # Step 1: Initialize VAR system [F_t, Y_t]
    # State dimension: r factors + 1 policy rate
    n_state = r + 1
    
    # VAR(1) coefficient matrix
    Phi = np.array([
        [0.8, 0.1, 0.0, -0.2],  # Factor 1 (real activity): responds negatively to policy
        [0.1, 0.7, 0.1, -0.1],  # Factor 2 (inflation): responds negatively to policy
        [0.0, 0.1, 0.6,  0.1],  # Factor 3 (financial): responds positively to policy
        [0.2, 0.3, 0.0,  0.9],  # Policy rate: responds to factors (Taylor rule)
    ])
    
    # Innovation covariance (structural shocks)
    Sigma_structural = np.diag([0.5, 0.4, 0.6, 0.3])  # Policy shock has std 0.3
    
    # Step 2: Simulate VAR
    Z = np.zeros((T, n_state))  # [F_1, F_2, F_3, Policy]
    Z[0, :] = np.random.randn(n_state) * 0.5
    
    for t in range(1, T):
        innovation = Sigma_structural @ np.random.randn(n_state)
        Z[t, :] = Phi @ Z[t-1, :] + innovation
    
    F_true = Z[:, :r]  # Factors
    Y_policy = Z[:, r]  # Policy rate
    
    # Step 3: Generate observed macro variables
    # Variables 0-19: Real activity (load on F1)
    # Variables 20-39: Prices/inflation (load on F2)
    # Variables 40-49: Financial (load on F3)
    
    Lambda_F = np.zeros((N, r))
    Lambda_Y = np.zeros((N, 1))
    
    # Real activity variables
    Lambda_F[:20, 0] = np.random.uniform(0.7, 1.0, 20)
    Lambda_F[:20, 1:] = np.random.uniform(-0.2, 0.3, (20, r-1))
    Lambda_Y[:20, 0] = np.random.uniform(-0.1, 0.1, 20)  # Small direct effect
    
    # Price variables
    Lambda_F[20:40, 1] = np.random.uniform(0.7, 1.0, 20)
    Lambda_F[20:40, [0, 2]] = np.random.uniform(-0.2, 0.3, (20, 2))
    Lambda_Y[20:40, 0] = np.random.uniform(-0.1, 0.1, 20)
    
    # Financial variables
    Lambda_F[40:, 2] = np.random.uniform(0.7, 1.0, 10)
    Lambda_F[40:, :2] = np.random.uniform(-0.2, 0.3, (10, 2))
    Lambda_Y[40:, 0] = np.random.uniform(0.2, 0.5, 10)  # Stronger direct effect
    
    # Generate observations
    X = (F_true @ Lambda_F.T + 
         Y_policy[:, np.newaxis] @ Lambda_Y.T + 
         np.random.randn(T, N) * 0.3)
    
    # Variable names
    var_names = (
        [f'Real_Activity_{i+1}' for i in range(20)] +
        [f'Price_{i+1}' for i in range(20)] +
        [f'Financial_{i+1}' for i in range(10)]
    )
    
    # Dates
    dates = pd.date_range('1990-01-01', periods=T, freq='M')
    
    return X, Y_policy, F_true, var_names, dates


# Generate FAVAR data
T, N, r_true = 300, 50, 3
X_panel, policy_rate, F_true, var_names, dates = generate_favar_data(T=T, N=N, r=r_true)

print(f"Generated FAVAR dataset:")
print(f"  Time periods (T): {T}")
print(f"  Macro variables (N): {N}")
print(f"  True factors (r): {r_true}")
print(f"  Policy variable: Federal Funds Rate")
print(f"\nVariable categories:")
print(f"  Real activity: {sum(1 for v in var_names if 'Real' in v)}")
print(f"  Prices:        {sum(1 for v in var_names if 'Price' in v)}")
print(f"  Financial:     {sum(1 for v in var_names if 'Financial' in v)}")

### Visualize Data

In [ ]:
# Purpose: Visualize policy rate and sample variables
# Key Concept: Policy shocks propagate through factors to all variables

fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# Plot 1: Policy rate
axes[0].plot(dates, policy_rate, linewidth=2, color='darkred')
axes[0].axhline(0, color='black', linestyle='--', linewidth=0.8)
axes[0].set_ylabel('Fed Funds Rate', fontsize=12)
axes[0].set_title('Observed Policy Variable', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Plot 2: Sample real activity variables
for i in [0, 5, 10]:
    axes[1].plot(dates, X_panel[:, i], alpha=0.7, linewidth=1.5, 
                 label=var_names[i])
axes[1].axhline(0, color='black', linestyle='--', linewidth=0.8)
axes[1].set_ylabel('Value', fontsize=12)
axes[1].set_title('Sample Real Activity Variables', fontsize=13, fontweight='bold')
axes[1].legend(loc='upper left', ncol=3, fontsize=10)
axes[1].grid(True, alpha=0.3)

# Plot 3: Sample price variables
for i in [20, 25, 30]:
    axes[2].plot(dates, X_panel[:, i], alpha=0.7, linewidth=1.5, 
                 label=var_names[i])
axes[2].axhline(0, color='black', linestyle='--', linewidth=0.8)
axes[2].set_xlabel('Date', fontsize=12)
axes[2].set_ylabel('Value', fontsize=12)
axes[2].set_title('Sample Price Variables', fontsize=13, fontweight='bold')
axes[2].legend(loc='upper left', ncol=3, fontsize=10)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

# Part 2: FAVAR Estimation

## 2.1 Two-Step Estimation Procedure

**Step 1:** Extract factors from $X_t$ (treating $Y_t$ as exogenous)
- Use PCA on $X_t$ to get $\hat{F}_t$
- Or use "slow-moving" factors that exclude contemporaneous $Y_t$

**Step 2:** Estimate VAR on $[\hat{F}_t, Y_t]$
- Standard VAR estimation via OLS
- Apply structural identification

In [ ]:
# Purpose: Implement two-step FAVAR estimation
# Key Concept: Factor extraction followed by VAR estimation

class FAVAR:
    """
    Factor-Augmented Vector Autoregression model.
    
    Two-step estimation:
    1. Extract factors from X
    2. Estimate VAR on [F, Y]
    """
    
    def __init__(self, n_factors=3, n_lags=4):
        """
        Parameters
        ----------
        n_factors : int
            Number of factors to extract
        n_lags : int
            VAR lag order
        """
        self.n_factors = n_factors
        self.n_lags = n_lags
        self.Phi_ = None      # VAR coefficient matrix
        self.Sigma_ = None    # VAR innovation covariance
        self.Lambda_F_ = None # Factor loadings
        self.F_ = None        # Estimated factors
    
    def fit(self, X, Y, standardize=True):
        """
        Estimate FAVAR model.
        
        Parameters
        ----------
        X : ndarray, shape (T, N)
            Panel of informational variables
        Y : ndarray, shape (T,) or (T, m)
            Observed policy/structural variables
        standardize : bool
            Whether to standardize X before PCA
        
        Returns
        -------
        self
        """
        T, N = X.shape
        
        # Ensure Y is 2D
        if Y.ndim == 1:
            Y = Y[:, np.newaxis]
        m = Y.shape[1]
        
        # Step 1: Extract factors via PCA
        if standardize:
            X_std = (X - X.mean(axis=0)) / X.std(axis=0)
        else:
            X_std = X
        
        pca = PCA(n_components=self.n_factors)
        self.F_ = pca.fit_transform(X_std)
        self.Lambda_F_ = pca.components_.T
        
        # Step 2: Construct VAR system [F, Y]
        Z = np.column_stack([self.F_, Y])  # (T, r+m)
        n_vars = Z.shape[1]
        
        # Step 3: Estimate VAR via OLS
        # Z_t = c + Phi_1 Z_{t-1} + ... + Phi_p Z_{t-p} + u_t
        
        T_var = T - self.n_lags
        
        # Dependent variable: Z_t
        Y_var = Z[self.n_lags:, :]
        
        # Regressors: [1, Z_{t-1}, ..., Z_{t-p}]
        X_var = np.ones((T_var, 1 + n_vars * self.n_lags))
        for lag in range(1, self.n_lags + 1):
            start_col = 1 + (lag - 1) * n_vars
            end_col = 1 + lag * n_vars
            X_var[:, start_col:end_col] = Z[self.n_lags - lag:-lag, :]
        
        # OLS for each equation
        self.Phi_ = np.linalg.lstsq(X_var, Y_var, rcond=None)[0]
        
        # Residuals and covariance
        residuals = Y_var - X_var @ self.Phi_
        self.Sigma_ = (residuals.T @ residuals) / T_var
        
        return self
    
    def impulse_response(self, horizon=24, shock_size=1, shock_var=None, 
                        identification='cholesky'):
        """
        Compute impulse response functions.
        
        Parameters
        ----------
        horizon : int
            IRF horizon
        shock_size : float
            Size of shock (in std deviations)
        shock_var : int, optional
            Variable to shock (default: last variable)
        identification : str
            'cholesky' for recursive identification
        
        Returns
        -------
        irf : ndarray, shape (horizon, n_vars)
            Impulse responses for VAR variables
        """
        n_vars = self.Phi_.shape[1]
        
        if shock_var is None:
            shock_var = n_vars - 1  # Default: shock last variable
        
        # Step 1: Structural identification
        if identification == 'cholesky':
            # Cholesky decomposition: Sigma = A A'
            A = np.linalg.cholesky(self.Sigma_)
        else:
            raise ValueError(f"Unknown identification: {identification}")
        
        # Step 2: Initial shock (one std dev to specified variable)
        shock = np.zeros(n_vars)
        shock[shock_var] = shock_size * np.sqrt(self.Sigma_[shock_var, shock_var])
        
        # Transform to structural shock
        structural_shock = np.linalg.solve(A, shock)
        
        # Step 3: Compute IRF via recursive substitution
        irf = np.zeros((horizon, n_vars))
        
        # Companion form for VAR(p)
        # [Z_t, Z_{t-1}, ..., Z_{t-p+1}]' = Phi_companion [Z_{t-1}, ..., Z_{t-p}]'
        companion_size = n_vars * self.n_lags
        Phi_companion = np.zeros((companion_size, companion_size))
        
        # First block row: VAR coefficients
        for lag in range(self.n_lags):
            start_col = lag * n_vars
            end_col = (lag + 1) * n_vars
            Phi_companion[:n_vars, start_col:end_col] = self.Phi_[1 + start_col:1 + end_col, :].T
        
        # Identity blocks for lags
        if self.n_lags > 1:
            Phi_companion[n_vars:, :-n_vars] = np.eye(companion_size - n_vars)
        
        # Initial state
        state = np.zeros(companion_size)
        state[:n_vars] = A @ structural_shock
        
        # Iterate forward
        for h in range(horizon):
            irf[h, :] = state[:n_vars]
            state = Phi_companion @ state
        
        return irf


print("FAVAR class defined successfully!")

### Exercise 2.1: Estimate FAVAR Model

**Task:** Fit a FAVAR model to the simulated data and examine the VAR coefficients.

**Expected Output:** VAR system should capture persistence and policy transmission.

In [ ]:
# YOUR CODE HERE
# ---------------
# Task: Fit FAVAR model and examine VAR structure
#
# Steps:
# 1. Create FAVAR object with n_factors=3, n_lags=4
# 2. Fit to X_panel and policy_rate
# 3. Examine extracted factors and VAR coefficients

# Step 1: Create model
favar = None  # Replace with FAVAR(n_factors=3, n_lags=4)

# Step 2: Fit model
# favar.fit(X_panel, policy_rate)

# Step 3: Examine results
print("\nFAVAR Estimation Results:")
print("=" * 60)
print(f"Extracted factors shape: {favar.F_.shape}")
print(f"VAR coefficient matrix shape: {favar.Phi_.shape}")
print(f"VAR innovation covariance shape: {favar.Sigma_.shape}")
print("\nVAR innovation covariance (diagonal):")
print(np.diag(favar.Sigma_).round(4))
print("=" * 60)

<details>
<summary>Hint: Model Creation</summary>
Use: favar = FAVAR(n_factors=3, n_lags=4)
</details>

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------
def test_exercise_2_1():
    assert favar is not None, "Don't forget to create FAVAR model!"
    assert favar.F_ is not None, "Model should be fitted!"
    assert favar.F_.shape == (T, 3), "Should extract 3 factors!"
    assert favar.Phi_ is not None, "VAR should be estimated!"
    
    # VAR system has 4 variables (3 factors + 1 policy)
    n_vars = 4
    # With 4 lags: 1 intercept + 4*4=16 lag coefficients
    assert favar.Phi_.shape[0] == 1 + n_vars * favar.n_lags, "VAR coefficient matrix wrong size!"
    
    # Check covariance is positive definite
    eigenvalues = np.linalg.eigvals(favar.Sigma_)
    assert np.all(eigenvalues > 0), "Covariance should be positive definite!"
    
    print("✓ Exercise 2.1 passed! FAVAR model estimated successfully.")

test_exercise_2_1()

In [ ]:
# SOLUTION
# --------
favar_sol = FAVAR(n_factors=3, n_lags=4)
favar_sol.fit(X_panel, policy_rate)

---

# Part 3: Structural Identification and IRFs

## 3.1 Monetary Policy Shock Identification

We'll use **recursive (Cholesky) identification** with ordering:
1. Factor 1 (real activity) - slow moving
2. Factor 2 (inflation) - slow moving
3. Factor 3 (financial) - fast moving
4. **Policy rate - responds contemporaneously to all**

This means:
- Policy shock affects all variables contemporaneously
- Real activity and inflation don't respond within period to policy
- Financial variables can respond within period

In [ ]:
# Purpose: Compute IRFs to monetary policy shock
# Key Concept: One std dev tightening shock to policy rate

# Compute IRF to policy shock (last variable in VAR system)
horizon = 24  # 2 years
irf_policy_shock = favar.impulse_response(
    horizon=horizon, 
    shock_size=1,  # One std dev shock
    shock_var=3,   # Policy rate (4th variable in [F1, F2, F3, Policy])
    identification='cholesky'
)

print(f"\nMonetary Policy Shock IRF:")
print(f"  Shock: 1 std dev tightening (increase in Fed Funds rate)")
print(f"  Horizon: {horizon} periods")
print(f"  IRF shape: {irf_policy_shock.shape}")
print(f"\nImpact effects (t=0):")
print(f"  Factor 1 (Real Activity): {irf_policy_shock[0, 0]:.4f}")
print(f"  Factor 2 (Inflation):     {irf_policy_shock[0, 1]:.4f}")
print(f"  Factor 3 (Financial):     {irf_policy_shock[0, 2]:.4f}")
print(f"  Policy Rate:              {irf_policy_shock[0, 3]:.4f}")

### Visualize Factor IRFs

In [ ]:
# Purpose: Visualize impulse responses of factors and policy rate
# Key Concept: Contractionary shock reduces activity and inflation over time

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
periods = np.arange(horizon)

# Plot 1: Factor 1 (Real Activity)
axes[0, 0].plot(periods, irf_policy_shock[:, 0], linewidth=2.5, color='steelblue')
axes[0, 0].axhline(0, color='black', linestyle='--', linewidth=0.8)
axes[0, 0].fill_between(periods, 0, irf_policy_shock[:, 0], alpha=0.3, color='steelblue')
axes[0, 0].set_xlabel('Periods', fontsize=11)
axes[0, 0].set_ylabel('Response', fontsize=11)
axes[0, 0].set_title('Factor 1: Real Activity', fontsize=12, fontweight='bold')
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: Factor 2 (Inflation)
axes[0, 1].plot(periods, irf_policy_shock[:, 1], linewidth=2.5, color='coral')
axes[0, 1].axhline(0, color='black', linestyle='--', linewidth=0.8)
axes[0, 1].fill_between(periods, 0, irf_policy_shock[:, 1], alpha=0.3, color='coral')
axes[0, 1].set_xlabel('Periods', fontsize=11)
axes[0, 1].set_ylabel('Response', fontsize=11)
axes[0, 1].set_title('Factor 2: Inflation', fontsize=12, fontweight='bold')
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: Factor 3 (Financial)
axes[1, 0].plot(periods, irf_policy_shock[:, 2], linewidth=2.5, color='darkgreen')
axes[1, 0].axhline(0, color='black', linestyle='--', linewidth=0.8)
axes[1, 0].fill_between(periods, 0, irf_policy_shock[:, 2], alpha=0.3, color='darkgreen')
axes[1, 0].set_xlabel('Periods', fontsize=11)
axes[1, 0].set_ylabel('Response', fontsize=11)
axes[1, 0].set_title('Factor 3: Financial', fontsize=12, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)

# Plot 4: Policy Rate
axes[1, 1].plot(periods, irf_policy_shock[:, 3], linewidth=2.5, color='darkred')
axes[1, 1].axhline(0, color='black', linestyle='--', linewidth=0.8)
axes[1, 1].fill_between(periods, 0, irf_policy_shock[:, 3], alpha=0.3, color='darkred')
axes[1, 1].set_xlabel('Periods', fontsize=11)
axes[1, 1].set_ylabel('Response', fontsize=11)
axes[1, 1].set_title('Policy Rate (Fed Funds)', fontsize=12, fontweight='bold')
axes[1, 1].grid(True, alpha=0.3)

fig.suptitle('Impulse Responses to Monetary Policy Shock (1 std dev tightening)', 
             fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

# Summary statistics
print(f"\nIRF Summary:")
print(f"  Peak negative effect on Real Activity: {irf_policy_shock[:, 0].min():.4f} at period {irf_policy_shock[:, 0].argmin()}")
print(f"  Peak negative effect on Inflation:     {irf_policy_shock[:, 1].min():.4f} at period {irf_policy_shock[:, 1].argmin()}")
print(f"  Cumulative effect on Real Activity:    {irf_policy_shock[:, 0].sum():.4f}")

## 3.2 Translate Factor IRFs to Observable Variables

The IRFs above show responses of **factors**. To understand the impact on **specific variables**, we use the loading matrix:

$$IRF_{X_i}(h) = \lambda_i' \cdot IRF_F(h)$$

where $\lambda_i$ is variable $i$'s loading on factors.

In [ ]:
# Purpose: Compute IRFs for all observable variables
# Key Concept: Factor loadings map factor IRFs to variable IRFs

def compute_observable_irfs(favar_model, factor_irfs, X_panel):
    """
    Compute IRFs for all observable variables.
    
    Parameters
    ----------
    favar_model : FAVAR
        Fitted FAVAR model
    factor_irfs : ndarray, shape (horizon, n_factors + m)
        Factor IRFs from VAR
    X_panel : ndarray, shape (T, N)
        Original panel data
    
    Returns
    -------
    observable_irfs : ndarray, shape (horizon, N)
        IRFs for all observable variables
    """
    horizon, n_state = factor_irfs.shape
    n_factors = favar_model.n_factors
    N = X_panel.shape[1]
    
    # Extract factor IRFs (first n_factors columns)
    factor_only_irfs = factor_irfs[:, :n_factors]
    
    # Extract policy IRF (last column)
    policy_irf = factor_irfs[:, -1]
    
    # Need to estimate Lambda_Y (loading of observables on policy)
    # Approximate: regress X on [F, Y]
    F = favar_model.F_
    Y = policy_rate[:, np.newaxis]
    
    X_std = (X_panel - X_panel.mean(axis=0)) / X_panel.std(axis=0)
    
    # Compute loadings for each variable
    Lambda_combined = np.zeros((N, n_factors + 1))
    for i in range(N):
        X_reg = np.column_stack([F, Y])
        beta = np.linalg.lstsq(X_reg, X_std[:, i], rcond=None)[0]
        Lambda_combined[i, :] = beta
    
    # Compute IRFs: X_i(h) = Lambda_i' * [F(h), Y(h)]'
    observable_irfs = np.zeros((horizon, N))
    for h in range(horizon):
        combined_state = np.concatenate([factor_only_irfs[h, :], [policy_irf[h]]])
        observable_irfs[h, :] = Lambda_combined @ combined_state
    
    return observable_irfs


# Compute IRFs for all observables
observable_irfs = compute_observable_irfs(favar, irf_policy_shock, X_panel)

print(f"\nObservable IRFs computed:")
print(f"  Shape: {observable_irfs.shape}")
print(f"  ({horizon} periods x {N} variables)")

### Visualize IRFs for Sample Variables

In [ ]:
# Purpose: Show policy shock impact on specific macro variables
# Key Concept: Different variables respond differently based on factor loadings

fig, axes = plt.subplots(3, 1, figsize=(14, 10))
periods = np.arange(horizon)

# Plot 1: Real activity variables
real_vars = [0, 5, 10]  # Sample from first 20
for i in real_vars:
    axes[0].plot(periods, observable_irfs[:, i], linewidth=2, label=var_names[i])
axes[0].axhline(0, color='black', linestyle='--', linewidth=0.8)
axes[0].set_ylabel('Response (std units)', fontsize=11)
axes[0].set_title('Real Activity Variables', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Plot 2: Price variables
price_vars = [20, 25, 30]  # Sample from 20-39
for i in price_vars:
    axes[1].plot(periods, observable_irfs[:, i], linewidth=2, label=var_names[i])
axes[1].axhline(0, color='black', linestyle='--', linewidth=0.8)
axes[1].set_ylabel('Response (std units)', fontsize=11)
axes[1].set_title('Price Variables', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

# Plot 3: Financial variables
financial_vars = [40, 45, 49]  # Sample from 40-49
for i in financial_vars:
    axes[2].plot(periods, observable_irfs[:, i], linewidth=2, label=var_names[i])
axes[2].axhline(0, color='black', linestyle='--', linewidth=0.8)
axes[2].set_xlabel('Periods', fontsize=11)
axes[2].set_ylabel('Response (std units)', fontsize=11)
axes[2].set_title('Financial Variables', fontsize=12, fontweight='bold')
axes[2].legend(fontsize=10)
axes[2].grid(True, alpha=0.3)

fig.suptitle('Observable Variable Responses to Monetary Policy Shock', 
             fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

print(f"\nObservation: Contractionary monetary policy shock (rate increase) leads to:")
print(f"  - Decline in real activity variables (negative IRFs)")
print(f"  - Decline in price variables (disinflationary effect)")
print(f"  - Mixed effects on financial variables (some rise initially)")

### Exercise 3.1: Create IRF Heatmap

**Task:** Create a heatmap showing IRFs for all 50 variables over time.

**Expected Output:** Clear pattern of negative responses for real/price variables.

In [ ]:
# YOUR CODE HERE
# ---------------
# Task: Create heatmap of all IRFs
#
# Use plt.imshow() to visualize observable_irfs.T

fig, ax = plt.subplots(figsize=(14, 10))

# Create heatmap (variables x time)
im = None  # Replace with ax.imshow(observable_irfs.T, ...)

# Customize plot
ax.set_xlabel('Periods After Shock', fontsize=12)
ax.set_ylabel('Variable Index', fontsize=12)
ax.set_title('IRF Heatmap: All Variables', fontsize=14, fontweight='bold')

# Add colorbar
# cbar = plt.colorbar(im, ax=ax, label='Response')

# Add group labels
ax.axhline(20, color='white', linewidth=2, linestyle='--')
ax.axhline(40, color='white', linewidth=2, linestyle='--')
ax.text(-2, 10, 'Real', rotation=90, va='center', fontsize=11, fontweight='bold')
ax.text(-2, 30, 'Prices', rotation=90, va='center', fontsize=11, fontweight='bold')
ax.text(-2, 45, 'Financial', rotation=90, va='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

<details>
<summary>Hint: Heatmap Creation</summary>
Use: im = ax.imshow(observable_irfs.T, aspect='auto', cmap='RdBu_r', interpolation='nearest')
The 'RdBu_r' colormap shows negative in blue, positive in red.
</details>

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------
def test_exercise_3_1():
    assert im is not None, "Don't forget to create heatmap!"
    
    # Check that figure exists
    assert plt.gcf().number > 0, "Figure should be created!"
    
    print("✓ Exercise 3.1 passed! IRF heatmap created.")

test_exercise_3_1()

In [ ]:
# SOLUTION
# --------
fig_sol, ax_sol = plt.subplots(figsize=(14, 10))
im_sol = ax_sol.imshow(observable_irfs.T, aspect='auto', cmap='RdBu_r', interpolation='nearest')
ax_sol.set_xlabel('Periods After Shock', fontsize=12)
ax_sol.set_ylabel('Variable Index', fontsize=12)
ax_sol.set_title('IRF Heatmap: All Variables', fontsize=14, fontweight='bold')
cbar_sol = plt.colorbar(im_sol, ax=ax_sol, label='Response')

---

# Summary and Key Takeaways

## What You've Learned

1. **FAVAR Framework**
   - Combines factor models with VAR for structural analysis
   - Incorporates information from 100+ variables in parsimonious system
   - Factors + observed policy variables jointly modeled
   - Avoids variable selection and curse of dimensionality

2. **Two-Step Estimation**
   - Step 1: Extract factors from large panel via PCA
   - Step 2: Estimate VAR on [Factors, Policy Variables]
   - Simple and computationally efficient
   - Joint estimation possible but more complex

3. **Structural Identification**
   - Cholesky (recursive) ordering based on speed of response
   - Policy shock ordered last (responds to all contemporaneously)
   - Alternative: sign restrictions, external instruments
   - Factor structure aids identification

4. **Impulse Response Analysis**
   - Compute IRFs for factors from VAR
   - Map to observable variables via loadings
   - Trace impact of shocks across entire economy
   - Quantify transmission mechanisms

## Key Empirical Findings (Bernanke et al., 2005)

**From original FAVAR paper on monetary policy:**
- Contractionary shock reduces GDP, employment, prices
- Effects peak after 12-18 months
- Financial variables respond immediately
- Price puzzle (prices rise after tightening) resolved with factors
- Factors capture Fed's information set better than small VAR

## FAVAR vs Standard VAR

| Feature | Standard VAR | FAVAR |
|---------|--------------|-------|
| Variables | 5-10 | 100+ (via factors) |
| Parameters | $K^2 p$ | $(r+m)^2 p$ (r << N) |
| Information | Limited | Comprehensive |
| IRF scope | Selected vars | All variables |
| Identification | Often difficult | Factor structure helps |

## Applications Beyond Monetary Policy

FAVAR useful for analyzing:
1. **Fiscal policy shocks** - Government spending impact
2. **Oil price shocks** - Energy price transmission
3. **Technology shocks** - Productivity innovations
4. **Financial shocks** - Credit crunch effects
5. **News shocks** - Anticipated vs surprise shocks

## Connection to Other Modules

| Module | Connection |
|--------|------------|
| Module 1-3 | Factor extraction foundation |
| Module 6.1 | Diffusion index forecasting |
| Module 7 | Sparse FAVAR with variable selection |
| Module 8 | Time-varying FAVAR |

## Extensions and Refinements

**To build production FAVAR system:**
1. **More variables:** Use full FRED-MD (120+ series)
2. **Stationarity:** Apply appropriate transformations
3. **Factor selection:** Use information criteria for r
4. **Lag selection:** Use BIC/AIC for VAR order p
5. **Uncertainty:** Bootstrap confidence bands for IRFs
6. **Alternative identification:** Sign restrictions, external IVs
7. **Forecast error variance decomposition:** Quantify shock contributions
8. **Historical decomposition:** Attribute movements to shocks

## Practical Applications

After this notebook, you can:
1. Estimate FAVAR models for structural analysis
2. Identify monetary policy and other structural shocks
3. Compute impulse response functions for factors and observables
4. Analyze shock transmission across many variables
5. Compare FAVAR to standard VAR
6. Apply to real-world policy questions

## Next Steps

You're now ready to:
1. Proceed to **Module 7: Sparse Methods** - Feature selection in high dimensions
2. Explore Module 8 advanced topics (time-varying parameters, non-Gaussian)
3. Complete capstone project with real FRED data

---

## Self-Assessment

Before moving on, ensure you can:
- [ ] Explain FAVAR framework and advantages over standard VAR
- [ ] Implement two-step FAVAR estimation
- [ ] Apply Cholesky identification for structural shocks
- [ ] Compute impulse response functions
- [ ] Map factor IRFs to observable variable IRFs
- [ ] Interpret monetary policy transmission results
- [ ] Create visualizations of IRFs across many variables

## Additional Resources

- **Bernanke et al. (2005):** "Measuring the Effects of Monetary Policy: A FAVAR Approach" - QJE (original paper)
- **Stock & Watson (2016):** "Dynamic Factor Models, Factor-Augmented VARs" - Handbook chapter
- **Banbura et al. (2010):** "Large Bayesian VARs" - JAE (Bayesian FAVAR)
- **Kilian & Lütkepohl (2017):** "Structural Vector Autoregressive Analysis" - Textbook

---

*You've successfully implemented FAVAR analysis! This powerful framework combines the dimensionality reduction of factor models with the structural interpretation of VARs, enabling rich analysis of shock transmission across the entire economy.*